In [1]:
import pandas as pd
import numpy as np

# Set random seed so our data stays consistent every time we run it
np.random.seed(42)

# Generate 150 unique SKUs
skus = ['SKU_' + str(i).zfill(3) for i in range(1, 151)]

# Generate realistic mock data for our warehouse
unit_cost = np.round(np.random.uniform(5, 500, 150), 2)          # Cost between $5 and $500
annual_demand = np.random.randint(50, 10000, 150)                # Yearly sales volume
lead_time_days = np.random.choice([7, 14, 21, 30, 45, 60], 150)  # Supplier delivery times
current_stock = np.round(annual_demand / 12 * np.random.uniform(0.5, 3.0, 150), 0) # Random stock levels

# Combine this into a Pandas DataFrame (a data table)
df = pd.DataFrame({
    'SKU': skus,
    'Unit_Cost': unit_cost,
    'Annual_Demand': annual_demand,
    'Lead_Time_Days': lead_time_days,
    'Current_Stock': current_stock
})

# Save this to a virtual CSV file we can use for our analysis
df.to_csv('inventory_data.csv', index=False)

print("Mock inventory dataset created successfully! Here are the first 5 SKUs:")
display(df.head())

Mock inventory dataset created successfully! Here are the first 5 SKUs:


,SKU,Unit_Cost,Annual_Demand,Lead_Time_Days,Current_Stock
0,SKU_001,190.40,7442,21,1281.0
1,SKU_002,475.60,6578,14,1398.0
2,SKU_003,367.34,5299,14,940.0
3,SKU_004,301.34,5222,14,442.0
4,SKU_005,82.23,1757,7,173.0


In [2]:
# 1. Calculate the total annual value for each SKU (Cost x Demand)
df['Annual_Value'] = df['Unit_Cost'] * df['Annual_Demand']

# 2. Sort the table from the highest value SKU to the lowest
df = df.sort_values(by='Annual_Value', ascending=False)

# 3. Calculate the running total (cumulative sum) as a percentage of the whole warehouse value
df['Cumulative_Value_%'] = df['Annual_Value'].cumsum() / df['Annual_Value'].sum()

# 4. Create a rule to assign A, B, or C based on that percentage
def assign_abc(percentage):
    if percentage <= 0.80:
        return 'A'  # Top 80% of the financial value
    elif percentage <= 0.95:
        return 'B'  # Next 15% of the financial value
    else:
        return 'C'  # Bottom 5% of the financial value

# 5. Apply the rule to create a new column called 'ABC_Class'
df['ABC_Class'] = df['Cumulative_Value_%'].apply(assign_abc)

# 6. Show the results
print("Count of SKUs in each class:")
display(df['ABC_Class'].value_counts())

print("\nTop 5 SKUs with their new classification:")
display(df[['SKU', 'Annual_Value', 'Cumulative_Value_%', 'ABC_Class']].head())

Count of SKUs in each class:


,count
ABC_Class,
A,67
C,45
B,38



Top 5 SKUs with their new classification:


,SKU,Annual_Value,Cumulative_Value_%,ABC_Class
69,SKU_070,4780631.37,0.027953,A
52,SKU_053,4351722.90,0.053398,A
88,SKU_089,4339096.73,0.078770,A
116,SKU_117,3820747.70,0.101111,A
12,SKU_013,3756876.48,0.123078,A


In [3]:
# 1. Generate a realistic volatility score (Coefficient of Variation) for each SKU
df['Demand_CV'] = np.round(np.random.uniform(0.1, 1.5, 150), 2)

# 2. Create a rule to assign X, Y, or Z based on how unpredictable the demand is
def assign_xyz(cv):
    if cv <= 0.5:
        return 'X'  # Steady, predictable demand
    elif cv <= 1.0:
        return 'Y'  # Fluctuating demand
    else:
        return 'Z'  # Highly unpredictable demand

# 3. Apply the rule to create a new column called 'XYZ_Class'
df['XYZ_Class'] = df['Demand_CV'].apply(assign_xyz)

# 4. Combine ABC and XYZ to get the final policy tag (like 'AX' or 'CZ')
df['ABC_XYZ_Class'] = df['ABC_Class'] + df['XYZ_Class']

# 5. Show the final breakdown
print("Inventory breakdown by ABC-XYZ Policy:")
display(df['ABC_XYZ_Class'].value_counts().sort_index())

print("\nTop 5 SKUs with their final classification:")
display(df[['SKU', 'ABC_Class', 'XYZ_Class', 'ABC_XYZ_Class']].head())

Inventory breakdown by ABC-XYZ Policy:


,count
ABC_XYZ_Class,
AX,18
AY,24
AZ,25
BX,9
BY,21
BZ,8
CX,11
CY,20
CZ,14



Top 5 SKUs with their final classification:


,SKU,ABC_Class,XYZ_Class,ABC_XYZ_Class
69,SKU_070,A,X,AX
52,SKU_053,A,Y,AY
88,SKU_089,A,Z,AZ
116,SKU_117,A,Y,AY
12,SKU_013,A,Z,AZ


In [4]:
# 1. Assign a Service Level (Z-Score) based on the ABC Class
# 'A' gets 98% protection (2.05), 'B' gets 95% (1.65), 'C' gets 90% (1.28)
def get_z_score(abc_class):
    if abc_class == 'A':
        return 2.05
    elif abc_class == 'B':
        return 1.65
    else:
        return 1.28

df['Z_Score'] = df['ABC_Class'].apply(get_z_score)

# 2. Calculate average daily sales and daily unpredictability (Standard Deviation)
df['Daily_Demand'] = df['Annual_Demand'] / 365
df['Daily_Volatility'] = df['Daily_Demand'] * df['Demand_CV']

# 3. Calculate the Optimal Safety Stock using the standard formula
# Formula: Z * Daily_Volatility * Square Root of Lead Time
df['Optimal_Safety_Stock'] = np.ceil(df['Z_Score'] * df['Daily_Volatility'] * np.sqrt(df['Lead_Time_Days']))

# 4. Show the results side-by-side
print("Safety Stock Calculated! Here are the top 5 SKUs:")
display(df[['SKU', 'ABC_XYZ_Class', 'Lead_Time_Days', 'Daily_Demand', 'Optimal_Safety_Stock']].head())

Safety Stock Calculated! Here are the top 5 SKUs:


,SKU,ABC_XYZ_Class,Lead_Time_Days,Daily_Demand,Optimal_Safety_Stock
69,SKU_070,AX,30,26.539726,99.0
52,SKU_053,AY,30,25.364384,194.0
88,SKU_089,AZ,45,26.764384,438.0
116,SKU_117,AY,7,25.986301,109.0
12,SKU_013,AZ,21,24.679452,344.0


In [5]:
# 1. Calculate the Lead Time Demand (Units sold while waiting for the truck)
df['Lead_Time_Demand'] = np.ceil(df['Daily_Demand'] * df['Lead_Time_Days'])

# 2. Calculate the true Reorder Point (ROP)
df['Reorder_Point'] = df['Lead_Time_Demand'] + df['Optimal_Safety_Stock']

# 3. Establish a Maximum Healthy Stock level
# (Assuming we order 30 days worth of cycle stock at a time)
df['Order_Quantity'] = np.ceil(df['Daily_Demand'] * 30)
df['Optimal_Max_Stock'] = df['Reorder_Point'] + df['Order_Quantity']

# 4. Find the Excess Units (Anything we hold above the Maximum Healthy Stock)
df['Excess_Units'] = df['Current_Stock'] - df['Optimal_Max_Stock']
# If the number is negative (meaning we are not overstocked), replace it with 0
df['Excess_Units'] = df['Excess_Units'].clip(lower=0)

# 5. Calculate the exact dollar amount tied up in this excess stock
df['Excess_Capital'] = df['Excess_Units'] * df['Unit_Cost']

# 6. Reveal the final business value!
total_cash_freed = df['Excess_Capital'].sum()
print(f"Total Working Capital Tied Up in Excess Stock: ${total_cash_freed:,.2f}")

print("\nTop 5 SKUs with their Financial Impact:")
display(df[['SKU', 'Current_Stock', 'Reorder_Point', 'Optimal_Max_Stock', 'Excess_Units', 'Excess_Capital']].head())

Total Working Capital Tied Up in Excess Stock: $2,368,287.44

Top 5 SKUs with their Financial Impact:


,SKU,Current_Stock,Reorder_Point,Optimal_Max_Stock,Excess_Units,Excess_Capital
69,SKU_070,2096.0,896.0,1693.0,403.0,198884.53
52,SKU_053,1548.0,955.0,1716.0,0.0,0.00
88,SKU_089,913.0,1643.0,2446.0,0.0,0.00
116,SKU_117,824.0,291.0,1071.0,0.0,0.00
12,SKU_013,1654.0,863.0,1604.0,50.0,20853.00
